In [1]:
from ml_workflow import (
    load_validate_data,
    train_evaluate_model,
    save_results
)

data_path = load_validate_data()

metrics = train_evaluate_model(data_path)

message = save_results(metrics)

print(message)

STEP 1: Loading Smart Campus Energy Dataset
Dataset loaded successfully!
Dataset shape: (2400, 22)

STEP 2: Preparing features
Training records: 1920
Testing records: 480

STEP 3: Training Logistic Regression
Model training completed!

STEP 4: Model Evaluation
accuracy: 0.8875
precision: 0.8010
recall: 0.9053
f1_score: 0.8500
roc_auc: 0.9662

Model saved at: C:\Users\eswar\MLOPS\deployment\exp6_orchestrated_model.pkl

Evaluation results saved at: C:\Users\eswar\MLOPS\deployment\exp6_results.json
ML workflow completed successfully!


In [3]:
import prefect

print("Prefect version:", prefect.__version__)

Prefect version: 3.9.1


In [4]:
from prefect import task, flow

@task(name="Load and Validate Dataset", log_prints=True)
def load_data_task():
    print("Loading and validating dataset...")
    return load_validate_data()


@task(name="Train and Evaluate Model", log_prints=True)
def train_model_task(data_path):
    print("Training machine learning model...")
    return train_evaluate_model(data_path)


@task(name="Save Evaluation Results", log_prints=True)
def save_results_task(metrics):
    print("Saving evaluation results...")
    return save_results(metrics)

print("Prefect tasks defined successfully!")

Prefect tasks defined successfully!


In [6]:
@flow(
    name="Smart Campus Energy ML Workflow",
    log_prints=True
)
def smart_campus_flow():

    data_path = load_data_task()

    metrics = train_model_task(data_path)

    result = save_results_task(metrics)

    print(result)

    return metrics

print("Prefect flow created successfully!")

Prefect flow created successfully!


In [7]:
prefect_results = smart_campus_flow()

print("\nFinal Prefect Workflow Results:")
print(prefect_results)

00:48:42.493 | INFO    | prefect - Starting temporary server on http://127.0.0.1:8512
See https://www.prefect.io/docs/v3/concepts/server#how-to-guides for more information on running a dedicated Prefect server.

00:49:24.652 | INFO    | Flow run 'garrulous-firefly' - Beginning flow run 'garrulous-firefly' for flow 'Smart Campus Energy ML Workflow'

00:49:25.881 | INFO    | Task run 'Load and Validate Dataset-aa6' - Loading and validating dataset...

00:49:25.886 | INFO    | Task run 'Load and Validate Dataset-aa6' - STEP 1: Loading Smart Campus Energy Dataset

00:49:25.925 | INFO    | Task run 'Load and Validate Dataset-aa6' - Dataset loaded successfully!

00:49:25.929 | INFO    | Task run 'Load and Validate Dataset-aa6' - Dataset shape: (2400, 22)

... (17 outputs left out)

00:49:26.326 | INFO    | Task run 'Save Evaluation Results-16c' - Finished in state Completed()

00:49:26.329 | INFO    | Flow run 'garrulous-firefly' - ML workflow completed successfully!

00:49:26.875 | INFO    | Flow run 'garrulous-firefly' - Finished in state Completed()


Final Prefect Workflow Results:
{'accuracy': 0.8875, 'precision': 0.8010471204188482, 'recall': 0.9053254437869822, 'f1_score': 0.85, 'roc_auc': 0.9661903765292339}


In [8]:
from pathlib import Path

airflow_dag_code = '''
from datetime import datetime
from airflow import DAG
from airflow.operators.python import PythonOperator

from ml_workflow import (
    load_validate_data,
    train_evaluate_model,
    save_results
)


def load_data_task():
    data_path = load_validate_data()
    return str(data_path)


def train_model_task(**context):
    data_path = context["ti"].xcom_pull(
        task_ids="load_validate"
    )
    metrics = train_evaluate_model(data_path)
    return metrics


def save_results_task(**context):
    metrics = context["ti"].xcom_pull(
        task_ids="train_evaluate"
    )
    return save_results(metrics)


with DAG(
    dag_id="smart_campus_energy_workflow",
    description="Smart Campus Energy ML Workflow",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
    tags=["MLOps", "Smart Campus"]
) as dag:

    task1 = PythonOperator(
        task_id="load_validate",
        python_callable=load_data_task
    )

    task2 = PythonOperator(
        task_id="train_evaluate",
        python_callable=train_model_task
    )

    task3 = PythonOperator(
        task_id="save_results",
        python_callable=save_results_task
    )

    task1 >> task2 >> task3
'''

dag_path = Path(
    r"C:\\Users\\eswar\\MLOPS\\airflow_project\\dags\\energy_workflow_dag.py"
)

dag_path.write_text(airflow_dag_code, encoding="utf-8")

print("Airflow DAG file created successfully!")
print("Location:", dag_path)

Airflow DAG file created successfully!
Location: C:\Users\eswar\MLOPS\airflow_project\dags\energy_workflow_dag.py
